## Activity 1 — Implement Entropy

In [1]:
import math

def entropy(labels):
    """
    Calculates the entropy of a list of labels.

    Args:
        labels (list): A list of categorical labels.

    Returns:
        float: The entropy of the labels.
    """
    # Count the frequency of each label
    label_counts = {}
    for label in labels:
        label_counts[label] = label_counts.get(label, 0) + 1

    total_labels = len(labels)
    if total_labels == 0:
        return 0.0

    # Calculate entropy
    entropy_value = 0.0
    for count in label_counts.values():
        probability = count / total_labels
        entropy_value -= probability * math.log2(probability)

    return entropy_value

# Test cases
labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

print(f"Entropy of labels1: {entropy(labels1):.4f}")
print(f"Entropy of labels2: {entropy(labels2):.4f}")
print(f"Entropy of labels3: {entropy(labels3):.4f}")


Entropy of labels1: 0.0000
Entropy of labels2: 1.0000
Entropy of labels3: 1.0000


### Explanation of Entropy Values

*   **`labels1 = ["Pass", "Pass", "Pass", "Pass"]`**
    *   **Entropy: 0.0**
    *   **Explanation:** In this set, all labels are identical ("Pass"). There is no uncertainty or randomness; if you pick a label, you are 100% certain it will be "Pass". Therefore, the entropy is 0.

*   **`labels2 = ["Pass", "Pass", "Fail", "Fail"]`**
    *   **Entropy: 1.0**
    *   **Explanation:** This set has two equally probable outcomes ("Pass" and "Fail"), each occurring 50% of the time. This represents the maximum possible entropy for a binary classification problem (log2(2) = 1 bit of information). There's maximum uncertainty about which label you'll get.

*   **`labels3 = ["Pass", "Fail", "Pass", "Fail"]`**
    *   **Entropy: 1.0**
    *   **Explanation:** Similar to `labels2`, this set also has two equally probable outcomes ("Pass" and "Fail"), each occurring 50% of the time. The order of the labels does not affect the overall probabilities and thus does not affect the entropy. It also represents maximum uncertainty for two classes.

## Activity 2 — Implement Mutual Information

In [2]:
from collections import defaultdict
import math

def conditional_entropy(labels, conditions):
    """
    Calculates the conditional entropy H(Labels | Conditions).

    Args:
        labels (list): A list of target labels.
        conditions (list): A list of conditions, corresponding to each label.

    Returns:
        float: The conditional entropy.
    """
    if len(labels) != len(conditions):
        raise ValueError("Labels and conditions lists must have the same length.")
    if not labels: # Handle empty lists
        return 0.0

    # Group labels by condition
    condition_label_groups = defaultdict(list)
    for label, condition in zip(labels, conditions):
        condition_label_groups[condition].append(label)

    total_conditional_entropy = 0.0
    total_samples = len(labels)

    for condition_value, sub_labels in condition_label_groups.items():
        # P(condition_value) - probability of this condition occurring
        prob_condition = len(sub_labels) / total_samples
        # H(Labels | condition_value) - entropy of labels given this condition
        entropy_given_condition = entropy(sub_labels)
        total_conditional_entropy += prob_condition * entropy_given_condition

    return total_conditional_entropy

def mutual_information(labels, features):
    """
    Calculates the Mutual Information I(Labels; Features) = H(Labels) - H(Labels | Features).

    Args:
        labels (list): A list of target labels.
        features (list): A list of feature values, corresponding to each label.

    Returns:
        float: The mutual information.
    """
    if len(labels) != len(features):
        raise ValueError("Labels and features lists must have the same length.")
    if not labels:
        return 0.0

    h_labels = entropy(labels)
    h_labels_given_features = conditional_entropy(labels, features)

    return h_labels - h_labels_given_features


# Example Dataset (from a typical Decision Tree scenario)
# Let's assume we have a dataset for student performance:
# Age, Study_Hours, Attendance, Pass/Fail
dataset = [
    {"Age": "Young", "Study_Hours": "Low", "Attendance": "Good", "Result": "Fail"},
    {"Age": "Young", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"},
    {"Age": "Young", "Study_Hours": "Low", "Attendance": "Poor", "Result": "Fail"},
    {"Age": "Adult", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"},
    {"Age": "Adult", "Study_Hours": "Low", "Attendance": "Good", "Result": "Fail"},
    {"Age": "Adult", "Study_Hours": "High", "Attendance": "Poor", "Result": "Pass"},
    {"Age": "Old", "Study_Hours": "Low", "Attendance": "Poor", "Result": "Fail"},
    {"Age": "Old", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"}
]

# Extract labels and features
results = [d["Result"] for d in dataset]
ages = [d["Age"] for d in dataset]
study_hours = [d["Study_Hours"] for d in dataset]
attendances = [d["Attendance"] for d in dataset]

# Calculate Mutual Information for each feature
mi_age = mutual_information(results, ages)
mi_study_hours = mutual_information(results, study_hours)
mi_attendance = mutual_information(results, attendances)

print(f"Mutual Information (Result; Age): {mi_age:.4f}")
print(f"Mutual Information (Result; Study_Hours): {mi_study_hours:.4f}")
print(f"Mutual Information (Result; Attendance): {mi_attendance:.4f}")


Mutual Information (Result; Age): 0.0613
Mutual Information (Result; Study_Hours): 1.0000
Mutual Information (Result; Attendance): 0.0488


## Activity 3 — Implement Prediction

### Manually Created Decision Tree Structure

Based on the Mutual Information calculations from Activity 2, 'Study_Hours' was the most informative feature. So, let's create a simple decision tree that first splits on 'Study_Hours'.

Our tree will be represented as a dictionary:
```
tree = {
    'feature': 'Study_Hours',
    'branches': {
        'High': 'Pass',  # If Study_Hours is High, predict Pass
        'Low': {
            'feature': 'Age', # If Study_Hours is Low, look at Age
            'branches': {
                'Young': 'Fail', # If Low Study_Hours and Young Age, predict Fail
                'Adult': 'Fail', # If Low Study_Hours and Adult Age, predict Fail
                'Old': 'Fail'    # If Low Study_Hours and Old Age, predict Fail
            }
        }
    }
}
```

In this simple tree, if 'Study_Hours' is 'High', the prediction is 'Pass'. If 'Study_Hours' is 'Low', it further checks 'Age', but for all 'Age' categories when 'Study_Hours' is 'Low', it predicts 'Fail' based on our limited dataset. (Note: A real ID3 algorithm would try to find the next best split, which here for 'Low Study_Hours' would still be 'Fail' as all instances with 'Low Study_Hours' resulted in 'Fail').

In [3]:
def predict(tree, sample):
    """
    Predicts the outcome for a given sample using the decision tree.

    Args:
        tree (dict): The decision tree structure.
        sample (dict): A dictionary representing the features of a new student.

    Returns:
        str: The predicted outcome (e.g., 'Pass' or 'Fail').
    """
    current_node = tree
    while isinstance(current_node, dict):
        feature = current_node['feature']
        value = sample.get(feature)

        if value in current_node['branches']:
            current_node = current_node['branches'][value]
        else:
            # Handle cases where the branch is not found (e.g., unseen feature value)
            # For simplicity, we can return a default or the most common class
            # Here, we'll assume a 'Fail' if an unknown path is taken.
            return 'Fail'

    return current_node

# Manually define the decision tree
tree = {
    'feature': 'Study_Hours',
    'branches': {
        'High': 'Pass',
        'Low': {
            'feature': 'Age',
            'branches': {
                'Young': 'Fail',
                'Adult': 'Fail',
                'Old': 'Fail'
            }
        }
    }
}

# Test with at least three new students
student1 = {"Age": "Adult", "Study_Hours": "High", "Attendance": "Good"}
student2 = {"Age": "Young", "Study_Hours": "Low", "Attendance": "Poor"}
student3 = {"Age": "Old", "Study_Hours": "High", "Attendance": "Good"}
student4 = {"Age": "Young", "Study_Hours": "High", "Attendance": "Poor"}
student5 = {"Age": "Adult", "Study_Hours": "Low", "Attendance": "Good"}

print(f"Student 1 (Study_Hours: High, Age: Adult) -> Predicted: {predict(tree, student1)}")
print(f"Student 2 (Study_Hours: Low, Age: Young) -> Predicted: {predict(tree, student2)}")
print(f"Student 3 (Study_Hours: High, Age: Old) -> Predicted: {predict(tree, student3)}")
print(f"Student 4 (Study_Hours: High, Age: Young) -> Predicted: {predict(tree, student4)}")
print(f"Student 5 (Study_Hours: Low, Age: Adult) -> Predicted: {predict(tree, student5)}")


Student 1 (Study_Hours: High, Age: Adult) -> Predicted: Pass
Student 2 (Study_Hours: Low, Age: Young) -> Predicted: Fail
Student 3 (Study_Hours: High, Age: Old) -> Predicted: Pass
Student 4 (Study_Hours: High, Age: Young) -> Predicted: Pass
Student 5 (Study_Hours: Low, Age: Adult) -> Predicted: Fail


## Activity 4 — Implement Recursive Training

In [4]:
from collections import defaultdict

def get_most_common_label(data, target_attribute_name):
    """
    Finds the most common label in a subset of data.
    """
    label_counts = defaultdict(int)
    for row in data:
        label_counts[row[target_attribute_name]] += 1
    if not label_counts:
        return None
    return max(label_counts, key=label_counts.get)

def train(data, features, target_attribute_name, max_depth=None, current_depth=0):
    """
    Recursively trains a decision tree using Mutual Information as the splitting criterion.

    Args:
        data (list of dict): The dataset (list of dictionaries, each representing a sample).
        features (list): A list of feature names available for splitting.
        target_attribute_name (str): The name of the target attribute (e.g., 'Result').
        max_depth (int, optional): The maximum depth of the tree. Defaults to None (no limit).
        current_depth (int): The current depth of the recursion. Used internally.

    Returns:
        dict or str: A dictionary representing the decision tree node,
                     or a string if it's a leaf node (predicted label).
    """

    # 1. Base Case: All samples have the same target value
    unique_labels = list(set([row[target_attribute_name] for row in data]))
    if len(unique_labels) == 1:
        return unique_labels[0]

    # 2. Base Case: No more features to split on or max depth reached
    if not features or (max_depth is not None and current_depth >= max_depth):
        return get_most_common_label(data, target_attribute_name)

    # 3. Find the best splitting feature
    best_feature = None
    max_mi = -1

    # Extract labels for MI calculation
    labels = [row[target_attribute_name] for row in data]

    for feature in features:
        feature_values = [row[feature] for row in data]
        mi = mutual_information(labels, feature_values)
        if mi > max_mi:
            max_mi = mi
            best_feature = feature

    # If no feature provides any information gain, return the most common label
    if best_feature is None or max_mi == 0:
        return get_most_common_label(data, target_attribute_name)

    # 4. Create a new tree node
    tree = {'feature': best_feature, 'branches': {}}

    # 5. Get unique values for the best feature
    unique_feature_values = list(set([row[best_feature] for row in data]))

    # 6. Recursively build subtrees for each branch
    remaining_features = [f for f in features if f != best_feature]
    for value in unique_feature_values:
        subset_data = [row for row in data if row[best_feature] == value]
        if not subset_data:
            # If a branch has no data, assign the most common label from the parent node
            tree['branches'][value] = get_most_common_label(data, target_attribute_name)
        else:
            tree['branches'][value] = train(
                subset_data,
                remaining_features,
                target_attribute_name,
                max_depth,
                current_depth + 1
            )

    return tree

# Define the dataset and features (from Activity 2)
dataset_for_training = [
    {"Age": "Young", "Study_Hours": "Low", "Attendance": "Good", "Result": "Fail"},
    {"Age": "Young", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"},
    {"Age": "Young", "Study_Hours": "Low", "Attendance": "Poor", "Result": "Fail"},
    {"Age": "Adult", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"},
    {"Age": "Adult", "Study_Hours": "Low", "Attendance": "Good", "Result": "Fail"},
    {"Age": "Adult", "Study_Hours": "High", "Attendance": "Poor", "Result": "Pass"},
    {"Age": "Old", "Study_Hours": "Low", "Attendance": "Poor", "Result": "Fail"},
    {"Age": "Old", "Study_Hours": "High", "Attendance": "Good", "Result": "Pass"}
]

all_features = ["Age", "Study_Hours", "Attendance"]
target_attribute = "Result"

# Train the decision tree
trained_tree = train(dataset_for_training, all_features, target_attribute)

# Print the trained tree for inspection
import json
print("\n--- Trained Decision Tree ---")
print(json.dumps(trained_tree, indent=4))



--- Trained Decision Tree ---
{
    "feature": "Study_Hours",
    "branches": {
        "Low": "Fail",
        "High": "Pass"
    }
}


## Activity 5 — Test the Trained Tree

In [5]:
# We will reuse the 'predict' function defined in Activity 3
# and the 'trained_tree' from Activity 4.

# Create at least five new student records
new_student1 = {"Age": "Young", "Study_Hours": "High", "Attendance": "Good"}
new_student2 = {"Age": "Adult", "Study_Hours": "Low", "Attendance": "Poor"}
new_student3 = {"Age": "Old", "Study_Hours": "High", "Attendance": "Good"}
new_student4 = {"Age": "Young", "Study_Hours": "Low", "Attendance": "Good"}
new_student5 = {"Age": "Adult", "Study_Hours": "High", "Attendance": "Poor"}
new_student6 = {"Age": "Old", "Study_Hours": "Low", "Attendance": "Good"}

print("--- Predictions using the Trained Decision Tree ---")
print(f"New Student 1 (Study_Hours: High) -> Predicted: {predict(trained_tree, new_student1)}")
print(f"New Student 2 (Study_Hours: Low) -> Predicted: {predict(trained_tree, new_student2)}")
print(f"New Student 3 (Study_Hours: High) -> Predicted: {predict(trained_tree, new_student3)}")
print(f"New Student 4 (Study_Hours: Low) -> Predicted: {predict(trained_tree, new_student4)}")
print(f"New Student 5 (Study_Hours: High) -> Predicted: {predict(trained_tree, new_student5)}")
print(f"New Student 6 (Study_Hours: Low) -> Predicted: {predict(trained_tree, new_student6)}")


--- Predictions using the Trained Decision Tree ---
New Student 1 (Study_Hours: High) -> Predicted: Pass
New Student 2 (Study_Hours: Low) -> Predicted: Fail
New Student 3 (Study_Hours: High) -> Predicted: Pass
New Student 4 (Study_Hours: Low) -> Predicted: Fail
New Student 5 (Study_Hours: High) -> Predicted: Pass
New Student 6 (Study_Hours: Low) -> Predicted: Fail
